# 在 Google Colab 上用 Qwen3 跑 OPD（On-Policy Distillation）

本 notebook 把本仓库的 **OPD（On-Policy Distillation）** 训练流程搬到 Google Colab，并使用 **Qwen3 系列**模型。

```
学生（可训练） : Qwen/Qwen3-0.6B
教师（冻结）   : Qwen/Qwen3-4B-Instruct-2507   ← 与本服务器 OPD 教师相同
训练数据       : DAPO-Math-17k（boxed 答案格式）
验证 / 评测    : MATH-500、AIME 2024、AIME 2025、AMC 2023
算法           : sampled-token OPD（opd.loss_type=sample_reverse_kl）
基础设施       : verl 0.7.1 + vLLM 0.12.0 + FSDP + Ray
```

OPD 的目标：学生自己 rollout 出轨迹，教师在同一段前缀上给出分布，学生最小化两者在**自己采样到的 token** 上的 KL：

```
L = mean_t  sg( log π_S(y_t|x,<t) − log π_T(y_t|x,<t) ) · log π_S(y_t|x,<t),   y_t ~ π_S
```

教师与学生共享同一份 `input_ids`，所以**两边必须使用同一个 tokenizer / 词表**。Qwen3 全系（0.6B/1.7B/4B/8B）共用词表，满足这一条件。Qwen 模型公开可下载，**不需要像 meta-llama 那样申请 gated 权限**。

---

## Colab 机型选择

| 机型 | GPU | 系统 RAM | 建议 |
|---|---|---|---|
| T4（免费） | 16 GB, sm75 | 12.7 GB | ❌ 不支持 bf16 / FlashAttention-2，4B 教师 + 优化器 offload 会 OOM |
| **L4（Pro）** | **24 GB, sm89** | **53 GB** | ✅ 本 notebook 默认配置（0.6B 学生 + 4B 教师） |
| **A100（Pro/Pro+）** | **40 GB, sm80** | **83.5 GB** | ✅ 更快；可把教师换成 `Qwen/Qwen3-8B-Instruct-2507` |
| A100 80GB / H100 | 80 GB | 167 GB | ✅ 可以加大 batch / 教师 |

预计耗时（L4）：安装 10–20 分钟（含 Python 3.12 venv），数据 3–5 分钟，smoke test 5–10 分钟，40 步训练约 2–3 小时（含两次验证）。

---

## 目录

1. GPU 检查与全局配置
2. 安装 verl + vLLM（在终端里执行，含可选 FlashAttention-2）
3. 获取代码、下载 Qwen3
4. 下载并构建 DAPO 训练集与评测集
5. Smoke test（2 步）
6. 正式训练（默认 40 步）
7. 用 vLLM 评测 checkpoint、汇总曲线
8. 保存结果到 Google Drive
9. 常见问题

> **重要**：Colab 默认 Python 是 **3.13**，而 verl 0.7.1 固定 `numpy<2`（`numpy==1.26.4` 没有 3.13 的 wheel）。所以第 2 节在**终端**里用 uv 建一个 **Python 3.12** 环境 `/content/verl_env`，训练 / 数据转换 / 评测都用它。装完**不用重启 Colab**，重新运行第 1 节即可。

## 1. GPU 检查与全局配置

这一节只做检查、选择模型和定义路径，不下载任何东西；在终端装完环境后需要重新运行一次（刷新 `VENV_DIR` / `PY`）。

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

import torch

# ============================================================
# 1.1 GPU 检查
# ============================================================
assert torch.cuda.is_available(), (
    "没有检测到 GPU。请在 Runtime -> Change runtime type 里选择 L4 GPU 或 A100 GPU。"
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
bf16_ok = torch.cuda.is_bf16_supported()

print(f"GPU     : {gpu_name}")
print(f"VRAM    : {vram_gb:.1f} GB")
print(f"bf16    : {bf16_ok}")
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__} (CUDA {torch.version.cuda})")

# 免费 T4 不建议运行；如果坚持要试，把这里改成 True（大概率 OOM）
ALLOW_T4 = False
if "T4" in gpu_name and not ALLOW_T4:
    raise SystemExit(
        "检测到 T4：Turing 架构不支持 bf16 和 FlashAttention-2，显存也放不下 "
        "4B 教师 + 0.6B 学生 + vLLM。请使用 Colab Pro 的 L4(24GB) 或 A100(40GB)。"
    )
if not bf16_ok:
    raise SystemExit("当前 GPU 不支持 bf16；本 notebook 的 verl 配置默认使用 bf16。")

# ============================================================
# 1.2 模型选择（教师 / 学生必须共用同一个 tokenizer）
# ============================================================
# Qwen 模型公开下载，不需要申请 gated 权限（Llama 的申请被拒也不影响）。
# 教师与本服务器 OPD 实验一致：Qwen3-4B-Instruct-2507（non-thinking）。
STUDENT_ID = "Qwen/Qwen3-0.6B"
TEACHER_ID = "Qwen/Qwen3-4B-Instruct-2507"

# Qwen3 thinking 模式开关。教师是 non-thinking 的 Instruct-2507，
# 学生必须同样 enable_thinking=False，否则教师/学生分布对不上。
ENABLE_THINKING = False

# A100 40GB / 80GB 想换更大教师时，直接覆盖：
# TEACHER_ID = "Qwen/Qwen3-8B-Instruct-2507"

# ============================================================
# 1.3 运行配置
# ============================================================
# 是否安装并启用 flash-attn（见第 2 节的可选单元）。False 时用 SDPA + padding。
USE_FLASH_ATTN = False

# vLLM 的 KV cache 占比。L4 24GB 用 0.35~0.40；A100 40GB 用 0.50~0.55。
GPU_MEM_UTIL = 0.35 if vram_gb < 30 else 0.50

# 教师（ref）的 FSDP 参数精度。教师没有 optimizer，bf16 既安全又省一半显存/内存。
# 学生（actor）保持仓库默认的 fp32 主权重（verl 的 mixed precision 会用 bf16 计算）。
REF_MODEL_DTYPE = "bf16"

# 正式训练的 batch / 步数（smoke test 单元会覆盖这些值）
TRAIN_BATCH_SIZE = 8
TOTAL_TRAINING_STEPS = 40

REPO_URL = "https://github.com/haha916/opd_test.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/OPSD_OnPolicyDistillation")
DATA_DIR = Path("/content/data")
MODEL_ROOT = Path("/content/models")
OUTPUT_ROOT = Path("/content/outputs")
HF_CACHE = Path("/content/hf_cache")

# 第 2 节在 Colab 终端里创建的 Python 3.12 环境（Colab 内核是 3.13，跑不了这个栈）
VENV_DIR = Path("/content/verl_env")
PY = VENV_DIR / "bin" / "python"

for p in (MODEL_ROOT, OUTPUT_ROOT, HF_CACHE):
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print()
print("Student          :", STUDENT_ID)
print("Teacher          :", TEACHER_ID)
print("VENV python      :", PY)
print("ENABLE_THINKING  :", ENABLE_THINKING)
print("USE_FLASH_ATTN   :", USE_FLASH_ATTN)
print("GPU_MEM_UTIL     :", GPU_MEM_UTIL)
print("REF_MODEL_DTYPE  :", REF_MODEL_DTYPE)
print("TRAIN_BATCH_SIZE :", TRAIN_BATCH_SIZE)
print("TOTAL_STEPS      :", TOTAL_TRAINING_STEPS)

## 2. 安装 verl + vLLM（在 Colab 终端里执行）

本仓库在以下组合上验证过，下面的 pin 不能省：

```
verl 0.7.1 / vllm 0.12.0 / torch 2.9.0+cu128 / transformers 4.57.1
numpy 1.26.4 / pyarrow 21.0.0 / datasets 5.0.1 / tensordict 0.10.0
```

- vLLM 0.12.0 固定 `torch==2.9.0`（PyPI 默认是 CUDA 12.8 build）。
- verl 固定 `numpy<2`，所以 numpy / scipy / opencv / pyarrow 都要用 NumPy 1.x 兼容版本。
- `verl==0.7.1` 是 `opd_trainer.yaml` 需要的最低版本（0.7.0 没有 `reward/reward` 配置）。

**为什么必须在终端里建 Python 3.12 环境：** Colab 当前默认 Python 是 **3.13**，而 `numpy==1.26.4`（verl 的 `numpy<2.0.0` 要求）没有 Python 3.13 的 wheel，源码编译也不支持 3.13。直接在 Colab 内核里 `pip install verl==0.7.1` 会失败。所以本 notebook 用 uv 建一个独立的 **Python 3.12** 环境 `/content/verl_env`：训练、数据转换、评测全部用这个环境里的 python；Colab 内核（3.13）只负责编排命令。

**打开终端**：Colab 左侧文件栏下方的 **Terminal** 图标，或菜单 **运行时 → 打开终端**。

安装耗时约 10–20 分钟（下载 4–6 GB），磁盘占用约 12–15 GB。


在终端里**整段复制执行**（不要在这个 notebook 单元里跑）：

```bash
# 0) 安装 uv（用来自动下载 Python 3.12 和装包）
pip install -q uv

# 1) 建 Python 3.12 虚拟环境（Colab 的 3.13 跑不了 numpy 1.26.4）
uv venv --seed /content/verl_env --python 3.12

# 2) 激活环境；之后的 pip / python 都是这个 3.12
source /content/verl_env/bin/activate

# 3) vLLM（会装 torch==2.9.0+cu128）
pip install -q "vllm==0.12.0"

# 4) verl 本体
pip install -q "verl==0.7.1"

# 5) 把关键包钉回仓库验证过的版本
pip install -q \
  "torch==2.9.0" \
  "transformers==4.57.1" \
  "datasets==5.0.1" \
  "pyarrow==21.0.0" \
  "numpy==1.26.4" \
  "scipy==1.16.3" \
  "opencv-python-headless==4.11.0.86" \
  "tensordict==0.10.0" \
  "ray==2.59.0" \
  "cachetools" \
  "hf_transfer"
```

验证（仍在终端、已激活环境）：

```bash
python -c "import torch, vllm, verl, numpy, transformers; \
print(torch.__version__, torch.version.cuda); print(vllm.__version__, verl.__version__, numpy.__version__, transformers.__version__)"
```

期望看到 `2.9.0+cu128`、`0.12.0`、`0.7.1`、`1.26.4`、`4.57.1`。

> uv 下载 Python 3.12 失败（网络问题）时，改用 micromamba：
> ```bash
> curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content bin/micromamba
> /content/bin/micromamba create -y -p /content/verl_env -c conda-forge python=3.12
> source /content/verl_env/bin/activate
> python -m pip install -U pip
> ```
> 然后从上面的第 3 步继续。


### 可选：安装 FlashAttention-2（L4 / A100）

默认走 **SDPA + padding**（`USE_REMOVE_PADDING=False, ATTN_IMPL=sdpa`），不装也能跑。想用仓库默认的 `flash_attention_2 + remove_padding` 就装下面这个**预编译 wheel**：

- **cp312 / torch 2.9.0 / CUDA 12.8**，和 `/content/verl_env` 完全匹配；支持 L4（sm89）和 A100（sm80）。
- 官方 FlashAttention wheel 只到 torch 2.8，所以用社区预编译包（本仓库 AutoDL 环境用的也是同一个 wheel，已验证 `flash_attn 2.8.3`）。
- 装好后把第 1 节的 `USE_FLASH_ATTN` 改成 `True`，**重新运行第 1 节**。

终端命令见下一单元。


```bash
source /content/verl_env/bin/activate

pip install -q "https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.0/flash_attn-2.8.3+cu128torch2.9-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl"

python -c "import flash_attn; print('flash_attn', flash_attn.__version__)"
```

wheel 约 242 MB（253,558,060 bytes），下载慢时可以给 URL 加 `https://gh-proxy.com/` 前缀。

> **不要用 Colab 默认的 Python 3.13 装**：这个 wheel 是 cp312；而且 verl 的 `numpy<2` 也要求整个训练栈跑在 3.12。
>
> 预编译 wheel 不可用时才考虑源码编译（需要 nvcc 12.8，耗时 1–2 小时）：
> ```bash
> source /content/verl_env/bin/activate
> nvcc --version   # 必须是 12.8
> pip install -q ninja psutil packaging
> MAX_JOBS=4 pip install flash-attn==2.8.3 --no-build-isolation
> ```


### 安装完成后：不用重启 Colab

- 环境装在 `/content/verl_env`，Colab 内核（Python 3.13）不受影响，所以**不需要 Restart session**。
- 回到第 1 节**重新运行一次**（刷新 `VENV_DIR` / `PY`），然后继续第 3 节。
- 之后如果又在终端里装 / 升级了包，训练前执行一次 `ray stop` 即可。


In [ ]:
import subprocess

assert PY.exists(), (
    f"找不到 {PY}。请先在 Colab 终端执行第 2 节的安装命令 "
    "(uv venv --seed /content/verl_env --python 3.12)。"
)

check = r'''
import torch, vllm, verl, transformers, datasets, ray, numpy
print("torch       ", torch.__version__, "| CUDA:", torch.version.cuda)
print("vllm        ", vllm.__version__)
print("verl        ", verl.__version__)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("numpy       ", numpy.__version__)
print("GPU         ", torch.cuda.get_device_name(0))
'''
subprocess.run([str(PY), "-c", check], check=True)

fa = subprocess.run(
    [str(PY), "-c", "import flash_attn; print(flash_attn.__version__)"],
    capture_output=True,
    text=True,
)
if fa.returncode == 0:
    print("flash_attn  ", fa.stdout.strip())
else:
    print("flash_attn   未安装（第 2 节可装；不装就用 SDPA）")


## 3. 获取代码、下载 Qwen3

### 代码怎么进 Colab？

这个 notebook 依赖仓库里**较新的 OPD 文件**（`scripts/opd/train_opd_sample.sh`、`src/opd/agent_loop_ext.py`、`src/opd/freeze.py`、`src/data/benchmark_utils.py`、`src/data/split_opd_train.py`、`scripts/eval/eval_ckpt_bench.py` 等）。如果这些文件还没推到 GitHub，`git clone` 下来的代码是不完整的，下面的预检会直接报错。两种解决方式：

**方式 A（推荐）：把当前代码推到 GitHub**

第 1 节的 `REPO_URL` 已经指向 `haha916/opd_test`（当前代码已上传）。以后改了代码，在 AutoDL 上这样更新：

```bash
cd /root/autodl-tmp/OPSD_OnPolicyDistillation
git add -A
git commit -m "colab qwen3 opd"
git remote add opd_test https://github.com/haha916/opd_test.git   # 只需一次
git push opd_test main
```

如果要换成别的仓库，再改第 1 节的 `REPO_URL` / `REPO_BRANCH`。

**方式 B：打包成 zip，经 Google Drive 上传**

在 AutoDL 上生成一个不含 outputs/.git 的压缩包：

```bash
cd /root/autodl-tmp/OPSD_OnPolicyDistillation
zip -r opd_colab_bundle.zip . \
  -x 'outputs/*' '.git/*' '**/__pycache__/*' '**/.ipynb_checkpoints/*' '*.pyc'
```

把 `opd_colab_bundle.zip` 上传到 Google Drive 的 `MyDrive/opd_colab_bundle.zip`，然后运行下面的「Drive 兜底」单元。

### HuggingFace 下载

- `Qwen/Qwen3-0.6B` 和 `Qwen/Qwen3-4B-Instruct-2507` 都是**公开模型，无需申请、无需 token**（Llama 的 gated 申请被拒也不影响）。
- 国内访问 huggingface.co 慢时，取消注释 `HF_ENDPOINT=https://hf-mirror.com`。
- 如果 Colab Secrets 里恰好有 `HF_TOKEN`，仍然会被使用（可提高限速配额）；没有也能正常下载。

In [ ]:
# ---- HuggingFace token（Qwen 公开模型不需要；有则用，可提高限速配额） ----
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("HF_TOKEN: loaded from Colab Secrets")
    else:
        print("HF_TOKEN: Colab Secret 为空（本地 Jupyter 请手动 export HF_TOKEN）")
except Exception:
    print("HF_TOKEN: 不在 Colab Secrets 中（本地 Jupyter 请手动 export HF_TOKEN）")

# 国内网络可取消注释：
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

# ---- clone repo ----
REQUIRED_FILES = [
    "scripts/opd/train_opd_sample.sh",
    "scripts/eval/eval_ckpt_bench.py",
    "src/data/benchmark_utils.py",
    "src/data/split_opd_train.py",
    "src/opd/agent_loop_ext.py",
    "src/opd/freeze.py",
    "src/rewards/math_reward.py",
]
REQUIRED_SNIPPETS = {
    "src/opd/losses.py": "sample_reverse_kl",
}


def ensure_repo():
    missing = [f for f in REQUIRED_FILES if not (REPO_DIR / f).exists()]
    if missing:
        raise SystemExit(
            "代码不完整，缺少以下文件：\n  "
            + "\n  ".join(missing)
            + "\n\n请二选一：\n"
            "  A. 在 AutoDL 上 commit + push 最新代码，并把第 1 节的 REPO_URL/REPO_BRANCH 改成你的仓库；\n"
            "  B. 按第 3 节 markdown 打包 opd_colab_bundle.zip 上传 Drive，然后运行下面的「Drive 兜底」单元。"
        )
    for rel, snippet in REQUIRED_SNIPPETS.items():
        if snippet not in (REPO_DIR / rel).read_text(encoding="utf-8"):
            raise SystemExit(f"{rel} 中没有 {snippet!r}，说明代码版本太旧，请更新代码。")


if not REPO_DIR.exists():
    print("cloning", REPO_URL, "branch", REPO_BRANCH)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print("repo already exists:", REPO_DIR)

ensure_repo()

sys.path.insert(0, str(REPO_DIR / "src"))

# 确认 OPD 模块能被导入（用 venv 的 Python；Colab 内核是 3.13，没有 verl）
check_env = os.environ.copy()
check_env["PYTHONPATH"] = str(REPO_DIR / "src")
subprocess.run(
    [str(PY), "-c", "from opd import losses; print('opd.losses imported OK')"],
    env=check_env,
    check=True,
)

head = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
)
print("repo:", REPO_DIR, "| HEAD:", head.stdout.strip() if head.returncode == 0 else "unknown (zip upload)")

### Drive 兜底：代码不完整时再运行

只有 `git clone` 下来的代码缺少上面那些新文件（预检报错）时才需要这个单元。它会挂载 Drive，用 `MyDrive/opd_colab_bundle.zip` 覆盖 `/content/OPSD_OnPolicyDistillation`，然后重新做预检。

打包命令见第 3 节 markdown（在 AutoDL 上执行）。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

bundle = Path("/content/drive/MyDrive/opd_colab_bundle.zip")
assert bundle.exists(), f"请先把 opd_colab_bundle.zip 上传到 Drive 的 MyDrive/（找不到 {bundle}）"

shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(["unzip", "-q", str(bundle), "-d", str(REPO_DIR)], check=True)
print("unzipped ->", REPO_DIR)

ensure_repo()

check_env = os.environ.copy()
check_env["PYTHONPATH"] = str(REPO_DIR / "src")
subprocess.run(
    [str(PY), "-c", "from opd import losses; print('opd.losses imported OK')"],
    env=check_env,
    check=True,
)

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer


def download_model(model_id: str) -> str:
    local_dir = MODEL_ROOT / model_id.split("/")[-1]
    if (local_dir / "config.json").exists():
        print(f"[skip]     {model_id} -> {local_dir}")
        return str(local_dir)
    print(f"[download] {model_id} -> {local_dir}")
    snapshot_download(
        repo_id=model_id,
        local_dir=str(local_dir),
        ignore_patterns=["*.bin", "*.pth", "original/*", "*.gguf"],
    )
    return str(local_dir)


STUDENT_MODEL = download_model(STUDENT_ID)
TEACHER_MODEL = download_model(TEACHER_ID)
print()
print("STUDENT_MODEL =", STUDENT_MODEL)
print("TEACHER_MODEL =", TEACHER_MODEL)

# ---- tokenizer 一致性检查（OPD 的硬性要求）----
tok_s = AutoTokenizer.from_pretrained(STUDENT_MODEL)
tok_t = AutoTokenizer.from_pretrained(TEACHER_MODEL)
print()
print(f"student vocab={len(tok_s)}, teacher vocab={len(tok_t)}, student eos={tok_s.eos_token_id}")

assert len(tok_s) == len(tok_t), "教师/学生词表大小不同，无法共享 input_ids"
assert tok_s.get_vocab() == tok_t.get_vocab(), "教师/学生词表内容不同，无法共享 input_ids"

sample = [{"role": "user", "content": "What is 1+1? Answer in \\boxed{}."}]
print("\n--- rendered Qwen3 chat prompt (enable_thinking=False) ---")
print(
    tok_s.apply_chat_template(
        sample, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )[-500:]
)

## 4. 下载并构建 DAPO 训练集与评测集

`prepare_grpo_data.py` 读的是**本地文件布局**（不是直接读 HF）。下面的单元从 HuggingFace 下载原始数据，转换成仓库期望的布局：

```
DATA_DIR/
├── DAPO-Math-17k/data/dapo-math-17k-dedup.parquet   ← 由下面的单元从 raw 文件去重得到
├── MATH-500/test.jsonl                  (problem, answer)
├── AIME_2024/test.parquet               (Problem, Answer, Solution)
├── AIME_2025/train.jsonl                (problem, answer)
└── AMC23/test.parquet                   (problem, answer)
```

> **为什么需要去重**：HuggingFace 上的 `BytedTsinghua-SIA/DAPO-Math-17k` 的 raw parquet 有 1,791,700 行，实际是 17,917 条 prompt 的 **100 份拼接**（17,917 × 100）。`prepare_grpo_data.py` 逐行处理，直接喂 raw 文件会得到 179 万条样本并让内存/时间爆掉。下面的单元会先按 prompt 去重，写出 `dapo-math-17k-dedup.parquet`（约 17.4k 条），脚本会优先读取它。

然后用 `prepare_grpo_data.py` 统一转成 verl 的 RL 格式（`prompt` 为 chat messages，`reward_model.ground_truth` 为答案），并用 `split_opd_train.py` 切出本次训练需要的 prompt 子集。

> Minerva / OlympiadBench 没有放进默认流程（`eval_ckpt_bench.py` 只支持上面四个 benchmark）。需要的话按仓库 README 的 Data Layout 放置 `Minerva/test.jsonl` 和 `OlympiadBench/test.parquet`。

In [ ]:
import json
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from datasets import load_dataset
from huggingface_hub import hf_hub_download

# ---------- DAPO-Math-17k（训练） ----------
dapo_dir = DATA_DIR / "DAPO-Math-17k"
dapo_dir.mkdir(parents=True, exist_ok=True)
dapo_path = hf_hub_download(
    repo_id="BytedTsinghua-SIA/DAPO-Math-17k",
    filename="data/dapo-math-17k.parquet",
    repo_type="dataset",
    local_dir=str(dapo_dir),
)
print("DAPO:", dapo_path)

# 官方 raw 文件是 100 份拼接（1,791,700 行 = 17,917 x 100）。
# prepare_grpo_data.py 逐行处理，必须先去重；DAPO_CANDIDATES 会优先读 dedup 文件。
dedup_path = dapo_dir / "data" / "dapo-math-17k-dedup.parquet"
if not dedup_path.exists():
    print("Deduplicating DAPO (raw file = 100 concatenated copies) ...")
    df = pq.read_table(dapo_path).to_pandas()
    print("  raw rows  :", len(df))
    df["_prompt_key"] = df["prompt"].astype(str)
    df = (
        df.drop_duplicates(subset=["_prompt_key"])
        .drop(columns=["_prompt_key"])
        .reset_index(drop=True)
    )
    print("  dedup rows:", len(df))
    df.to_parquet(dedup_path)
    print("wrote", dedup_path)


def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"wrote {path} ({len(rows)} rows)")


def write_parquet(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_parquet(path)
    print(f"wrote {path} ({len(rows)} rows)")


# ---------- MATH-500 ----------
ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
write_jsonl(
    DATA_DIR / "MATH-500" / "test.jsonl",
    [{"problem": r["problem"], "answer": r["answer"]} for r in ds],
)

# ---------- AIME 2024 ----------
ds = load_dataset("Maxwell-Jia/AIME_2024", split="train")
write_parquet(
    DATA_DIR / "AIME_2024" / "test.parquet",
    [
        {"Problem": r["Problem"], "Answer": r["Answer"], "Solution": r.get("Solution", "")}
        for r in ds
    ],
)

# ---------- AIME 2025 ----------
try:
    ds = load_dataset("MathArena/aime_2025", split="train")
    aime25 = [{"problem": r["problem"], "answer": r["answer"]} for r in ds]
except Exception as e:
    print("MathArena/aime_2025 失败，回退到 yentinglin/aime_2025:", e)
    ds = load_dataset("yentinglin/aime_2025", split="train")
    aime25 = [{"problem": r["problem"], "answer": r["answer"]} for r in ds]
write_jsonl(DATA_DIR / "AIME_2025" / "train.jsonl", aime25)

# ---------- AMC 2023 ----------
# AI-MO/aimo-validation-amc 含 AMC 2022+2023 共 83 题；仓库的 AMC23 基准只取 2023（40 题）。
ds = load_dataset("AI-MO/aimo-validation-amc", split="train")
rows = [
    {"problem": r["problem"], "answer": r["answer"]}
    for r in ds
    if "2023" in str(r.get("url", ""))
]
if not rows:  # url 字段格式变化时退回全量
    rows = [{"problem": r["problem"], "answer": r["answer"]} for r in ds]
write_parquet(DATA_DIR / "AMC23" / "test.parquet", rows)

In [ ]:
# ---------- 转成 verl RL 格式 ----------
PROCESSED = DATA_DIR / "grpo_processed"

cmd = [
    str(PY), str(REPO_DIR / "src/data/prepare_grpo_data.py"),
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(PROCESSED),
    "--train-ratio", "1.0",
    "--benchmarks", "math500", "aime24", "aime25", "amc23",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

# 奖励函数 sanity check（boxed 答案抽取 + 等价判断）
from rewards.math_reward import compute_score

assert compute_score("The answer is \\boxed{5}.", "5") == 1.0
assert compute_score("The answer is \\boxed{\\frac{1}{2}}.", "0.5") == 1.0
print("\nreward function sanity check: OK")

In [ ]:
# ---------- 切出本次 OPD 训练用的 prompt 子集 ----------
OPD_SIZE = TRAIN_BATCH_SIZE * TOTAL_TRAINING_STEPS

cmd = [
    str(PY), str(REPO_DIR / "src/data/split_opd_train.py"),
    "--input", str(PROCESSED / "train.parquet"),
    "--output-dir", str(PROCESSED),
    "--opd-size", str(OPD_SIZE),
    "--seed", "42",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

# 看一眼训练格式和 Qwen3 渲染后的 prompt
train_df = pd.read_parquet(PROCESSED / f"train_opd_{OPD_SIZE}.parquet")
print("\n--- raw prompt messages ---")
print(train_df.iloc[0]["prompt"])
print("\n--- rendered by Qwen3 student tokenizer (tail, enable_thinking=False) ---")
print(
    tok_s.apply_chat_template(
        train_df.iloc[0]["prompt"],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )[-700:]
)

## 5. Smoke test（2 步）

先用 16 条 prompt、2 个优化步把整条链路跑通：

```
DAPO parquet -> vLLM rollout -> teacher forward -> sample_reverse_kl -> 保存 HF checkpoint
```

这一节会覆盖成很小的配置：

- `TOTAL_TRAINING_STEPS=2`，`TRAIN_BATCH_SIZE=8`，`OPD_SIZE=16`
- 训练 rollout 上限 1024 tokens，验证上限 2048
- `TEST_FREQ=-1`（不做验证），`PLATEAU_ENABLE=False`
- `SAVE_FREQ=2`（第 2 步保存一个 HF checkpoint）
- `FREEZE_LM_HEAD=False`

> **Qwen3 注意**：Qwen3-0.6B 的 `tie_word_embeddings=true`，`lm_head` 与输入 embedding 共享同一个参数。设置 `FREEZE_LM_HEAD=True` 会**同时冻结输入 embedding**（脚本会打印 warning）。只有在显存不足时才建议打开；默认关闭。
>
> **thinking 模式**：教师 `Qwen3-4B-Instruct-2507` 是 non-thinking 模型，学生 rollout 也必须 `enable_thinking=False`。本 notebook 通过第 1 节的 `ENABLE_THINKING=False` 传给训练脚本；如果改成 True，脚本检测到教师是 Instruct 会直接报错退出。

先跑下面的「Colab 补丁」单元，再跑 smoke test。

In [ ]:
# Colab 补丁：给 train_opd_sample.sh 增加一个透传 Hydra 覆盖的入口。
# 用途：设置 data.dataloader_num_workers=0。Colab 的 /dev/shm 可能很小，
# 多进程 DataLoader 容易报 "Bus error"。
train_sh = REPO_DIR / "scripts/opd/train_opd_sample.sh"
src = train_sh.read_text()

if "EXTRA_HYDRA_ARGS" not in src:
    needle = "    trainer.total_training_steps=$total_training_steps\n"
    assert needle in src, "train_opd_sample.sh 的结构变了，请手动加上 ${EXTRA_HYDRA_ARGS}"
    src = src.replace(
        needle,
        "    trainer.total_training_steps=$total_training_steps \\\n    ${EXTRA_HYDRA_ARGS}\n",
    )
    train_sh.write_text(src)
    print("patched", train_sh)
else:
    print("already patched:", train_sh)

In [ ]:
def run_opd(**overrides):
    """Launch scripts/opd/train_opd_sample.sh with environment overrides."""
    assert PY.exists(), f"找不到 {PY}，请先在终端执行第 2 节的安装命令"
    env = os.environ.copy()
    env.update({
        "MODEL_PATH": STUDENT_MODEL,
        "TEACHER_MODEL_PATH": TEACHER_MODEL,
        "DATA_DIR": str(DATA_DIR),
        "OUTPUT_ROOT": str(OUTPUT_ROOT),
        "PATH": f"{VENV_DIR / 'bin'}:{env.get('PATH', '')}",
        "VIRTUAL_ENV": str(VENV_DIR),
        "PYTHONPATH": f"{REPO_DIR / 'src'}:{env.get('PYTHONPATH', '')}",
        "PYTORCH_ALLOC_CONF": "expandable_segments:True",
        "TOKENIZERS_PARALLELISM": "false",
        # Qwen3: 教师 non-thinking，学生也必须关闭 thinking
        "ENABLE_THINKING": str(ENABLE_THINKING),
        # 1) 关掉多进程 dataloader，避免 Colab /dev/shm 太小导致 Bus error
        # 2) 教师用 bf16 权重（无 optimizer，安全且省显存）
        "EXTRA_HYDRA_ARGS": (
            "data.dataloader_num_workers=0 "
            f"actor_rollout_ref.ref.fsdp_config.model_dtype={REF_MODEL_DTYPE}"
        ),
    })
    env.update({k: str(v) for k, v in overrides.items()})

    print("=== OPD launch ===")
    for k in sorted(overrides):
        print(f"  {k}={overrides[k]}")
    print("==================")

    subprocess.run(["bash", str(REPO_DIR / "scripts/opd/train_opd_sample.sh")], env=env, check=True)


run_opd(
    RUN_ID="smoke",
    TOTAL_TRAINING_STEPS=2,
    TOTAL_EPOCHS=1,
    TRAIN_BATCH_SIZE=8,
    PPO_MINI_BATCH_SIZE=8,
    PPO_MICRO_BATCH_SIZE_PER_GPU=1,
    OPD_TRAIN_SIZE=16,
    FORCE_RESPLIT=1,
    LEARNING_RATE=1e-6,
    MAX_PROMPT_LENGTH=1024,
    MAX_RESPONSE_LENGTH=2048,
    TRAIN_RESPONSE_LENGTH=1024,
    EVAL_RESPONSE_LENGTH=2048,
    TEST_FREQ=-1,
    PLATEAU_ENABLE=False,
    SAVE_FREQ=2,
    MAX_ACTOR_CKPT_TO_KEEP=1,
    FREEZE_LM_HEAD=False,
    USE_REMOVE_PADDING=USE_FLASH_ATTN,
    ATTN_IMPL="" if USE_FLASH_ATTN else "sdpa",
    GPU_MEM_UTIL=GPU_MEM_UTIL,
    VAL_N=2,
)

## 6. 正式训练（默认 40 步）

默认在 L4 上跑 **40 步 × 8 prompts = 320 条 DAPO prompt**：

- 训练 rollout 上限 2048 tokens，评测上限 4096
- 每 20 步在 AIME24 / AIME25 上验证一次（`VAL_N=4`）
- 第 20、40 步各保存一个 HuggingFace checkpoint（`MAX_ACTOR_CKPT_TO_KEEP=2`）

放大实验时改这些值：

| 变量 | 默认 | 说明 |
|---|---|---|
| `TRAIN_BATCH_SIZE` | 8 | L4 建议 8；A100 可 16 |
| `TOTAL_TRAINING_STEPS` | 40 | 100 / 200 更接近真实实验 |
| `TRAIN_RESPONSE_LENGTH` | 2048 | A100 可 4096 |
| `VAL_N` | 4 | 验证采样数，越大越稳但越慢 |
| `TEST_FREQ` | 20 | 验证频率 |
| `TEACHER_ID` | Qwen3-4B-Instruct-2507 | A100 40GB 可换 `Qwen/Qwen3-8B-Instruct-2507`（需要更多显存/内存） |
| `USE_FLASH_ATTN` | False | 先跑第 2 节的可选安装单元，再改成 True |

> AIME 对 0.6B 学生很难，前期准确率接近 0 是正常的。先看训练日志里的 **loss / KL / response length**，以及评测里的 **boxed_rate / trunc_rate**；想更快看到能力信号，可以在第 7 节把评测重点放在 MATH-500。

In [ ]:
TRAIN_BATCH_SIZE = 8            # L4: 8; A100: 16
TOTAL_TRAINING_STEPS = 40       # 320 prompts
OPD_SIZE = TRAIN_BATCH_SIZE * TOTAL_TRAINING_STEPS

run_opd(
    RUN_ID="qwen3-0.6b-opd",
    TOTAL_TRAINING_STEPS=TOTAL_TRAINING_STEPS,
    TOTAL_EPOCHS=1,
    TRAIN_BATCH_SIZE=TRAIN_BATCH_SIZE,
    PPO_MINI_BATCH_SIZE=TRAIN_BATCH_SIZE,
    PPO_MICRO_BATCH_SIZE_PER_GPU=1 if vram_gb < 30 else 2,
    OPD_TRAIN_SIZE=OPD_SIZE,
    FORCE_RESPLIT=1,
    LEARNING_RATE=1e-6,
    MAX_PROMPT_LENGTH=1024,
    MAX_RESPONSE_LENGTH=4096,
    TRAIN_RESPONSE_LENGTH=2048,
    EVAL_RESPONSE_LENGTH=4096,
    TEST_FREQ=20,
    VAL_N=4,
    VAL_TEMPERATURE=0.6,
    VAL_TOP_P=0.9,
    VAL_TOP_K=-1,
    PLATEAU_ENABLE=False,
    SAVE_FREQ=20,
    MAX_ACTOR_CKPT_TO_KEEP=2,
    FREEZE_LM_HEAD=False,
    USE_REMOVE_PADDING=USE_FLASH_ATTN,
    ATTN_IMPL="" if USE_FLASH_ATTN else "sdpa",
    GPU_MEM_UTIL=GPU_MEM_UTIL,
)

## 7. 用 vLLM 评测 checkpoint

训练脚本设置了 `actor_rollout_ref.actor.checkpoint.save_contents=["hf_model"]`，所以 checkpoint 是可直接被 vLLM 加载的 HuggingFace 权重：

```
outputs/<EXP_NAME>/global_step_20/actor/huggingface
outputs/<EXP_NAME>/global_step_40/actor/huggingface
```

`eval_ckpt_bench.py` 里写死的 stop token id（`[151645, 151643]`）正好是 Qwen 的 `<|im_end|>` / `<|endoftext|>`，**不需要像 Llama 那样打补丁**；下面的单元只做一次校验。

评测超参：`temperature=0.6, top_p=0.95, n=4, max_tokens=4096`，与本服务器 OPD 评测一致，并显式 `enable_thinking=False`。

In [ ]:
# 释放训练残留的 Ray / vLLM 进程
subprocess.run([str(VENV_DIR / "bin" / "ray"), "stop"], check=False)

# Qwen 的 stop token id 与 eval_ckpt_bench.py 写死的 [151645, 151643] 一致，无需打补丁。
# 这里做一次校验，防止将来换 tokenizer 后静默错位。
from transformers import AutoTokenizer

tok_eval = AutoTokenizer.from_pretrained(STUDENT_MODEL)
qwen_stop = sorted(
    {tok_eval.eos_token_id, tok_eval.convert_tokens_to_ids("<|endoftext|>")}
)
assert qwen_stop == [151643, 151645], f"意外的 Qwen stop ids: {qwen_stop}"
print("Qwen stop_token_ids OK:", qwen_stop)

# 找到本次训练的 output 目录和 checkpoint
EXP_DIRS = sorted(OUTPUT_ROOT.glob("*qwen3-0.6b-opd*"))
assert EXP_DIRS, "找不到训练输出目录；请先跑第 6 节的正式训练单元"
EXP_DIR = EXP_DIRS[-1]
ckpts = sorted(EXP_DIR.glob("global_step_*"))
assert ckpts, f"{EXP_DIR} 下没有 global_step_* checkpoint；检查 SAVE_FREQ / 训练日志。"
print("\nEXP_DIR    :", EXP_DIR)
print("checkpoints:", [c.name for c in ckpts])


In [ ]:
def eval_ckpt(ckpt_dir, tag, benchmarks=("math500", "aime24", "aime25", "amc23"),
              n=4, limit_prompts=100):
    """Evaluate one HF checkpoint with vLLM.

    limit_prompts: 每个 benchmark 只取前 N 题（0 = 全量）。快速看趋势时用 100。
    """
    out_dir = OUTPUT_ROOT / "eval" / tag
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        str(PY), str(REPO_DIR / "scripts/eval/eval_ckpt_bench.py"),
        "--model-path", str(ckpt_dir),
        "--tokenizer-path", STUDENT_MODEL,   # 固定用学生原始 tokenizer，保证 prompt 编码一致
        "--tag", tag,
        "--benchmarks", *benchmarks,
        "--data-dir", str(PROCESSED),
        "--out-dir", str(out_dir),
        "--n", str(n),
        "--max-tokens", "4096",
        "--temperature", "0.6",
        "--top-p", "0.95",
        "--no-enable-thinking",
        "--max-model-len", "6144",
        "--max-num-seqs", "64",
        "--gpu-memory-utilization", "0.85",
    ]
    if limit_prompts:
        cmd += ["--limit-prompts", str(limit_prompts)]

    print("\n" + " ".join(cmd))
    subprocess.run(cmd, check=True)


# 先评最后一个 checkpoint（limit_prompts=100，快速看趋势）
last = ckpts[-1]
last_tag = f"step_{last.name.split('_')[-1]}"
eval_ckpt(last / "actor" / "huggingface", tag=last_tag, limit_prompts=100)

# 评全部 checkpoint + 全量题目：
# for c in ckpts:
#     eval_ckpt(c / "actor" / "huggingface", f"step_{c.name.split('_')[-1]}", limit_prompts=0)

In [ ]:
# 汇总所有已完成的评测结果，并生成 checkpoint 曲线 / plateau 报告
eval_root = OUTPUT_ROOT / "eval"

subprocess.run(
    [
        str(PY), str(REPO_DIR / "scripts/eval/collect_eval_curve.py"),
        "--root", str(eval_root),
        "--benchmarks", "math500", "aime24", "aime25", "amc23",
    ],
    check=False,
)

subprocess.run(
    [
        str(PY), str(REPO_DIR / "scripts/eval/analyze_eval_curve.py"),
        "--root", str(eval_root),
        "--delta", "2.0",
        "--min-points", "2",
    ],
    check=False,
)

print("\nresults:", eval_root)
print("curve   :", eval_root / "curve.csv")
print("report  :", eval_root / "report.md")

## 8. 保存结果到 Google Drive（可选）

Colab 运行时是临时的。建议至少把 `outputs/eval`、训练日志（`*.json` / `*.log`）和 `curve.csv` 复制到 Drive。

模型权重每个约 1.5 GB（Qwen3-0.6B bf16），按需复制；不要每次都把 `global_step_*` 全量拷过去。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DEST = Path("/content/drive/MyDrive/opd_qwen3_runs")
DEST.mkdir(parents=True, exist_ok=True)

# 评测结果 + 曲线
if (OUTPUT_ROOT / "eval").exists():
    shutil.copytree(OUTPUT_ROOT / "eval", DEST / "eval", dirs_exist_ok=True)

# 训练日志（不复制权重）
for p in OUTPUT_ROOT.rglob("*.json"):
    rel = p.relative_to(OUTPUT_ROOT)
    target = DEST / "logs" / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, target)

# 确实需要保存 checkpoint 时再取消注释（0.6B 约 1.5GB/个）：
# shutil.copytree(EXP_DIR, DEST / EXP_DIR.name, dirs_exist_ok=True)

print("saved to", DEST)

## 9. 常见问题（Colab / Qwen3）

### CUDA OOM

按顺序尝试：

1. `TRAIN_BATCH_SIZE` 8 → 4；`PPO_MICRO_BATCH_SIZE_PER_GPU=1`
2. `MAX_RESPONSE_LENGTH` / `TRAIN_RESPONSE_LENGTH` 减半（例如 2048 → 1024）
3. `GPU_MEM_UTIL` 0.35 → 0.25（vLLM KV cache 变小）
4. `FREEZE_LM_HEAD=True`（注意：Qwen3-0.6B 的 embedding 与 lm_head 共享，会一起冻结）
5. 学生也切成 bf16 主权重：在 `run_opd` 的 `EXTRA_HYDRA_ARGS` 里加
   `actor_rollout_ref.actor.fsdp_config.model_dtype=bf16`
   （verl 官方注释建议 actor 用 fp32 建 optimizer；bf16 能省一半显存，但优化稳定性略差）
6. 教师暂时换成 `Qwen/Qwen3-0.6B` 只做链路 debug（与学生同权，没有蒸馏意义）

### Colab 默认 Python 3.13，为什么还要建 3.12 环境？

verl 0.7.1 的依赖写死 `numpy<2.0.0`，而 `numpy==1.26.4` 没有 cp313 wheel，源码编译也不支持 3.13。
所以第 2 节用 uv 建了 `/content/verl_env`（Python 3.12）。notebook 里所有数据转换 / 训练 / 评测
都用 `/content/verl_env/bin/python`（第 1 节的 `PY`），Colab 内核只做编排，不 import verl / vllm。

### `vllm` / `torch` 导入错误

- 训练栈在 `/content/verl_env`（Python 3.12），**不要**用 Colab 内核的 3.13 装 verl。
- 检查是否装进了正确的环境：`/content/verl_env/bin/python -c "import verl, vllm; print(verl.__version__, vllm.__version__)"`。
- 如果 `torch.version.cuda` 显示 12.6 而不是 12.8，且 vLLM 报 CUDA 错误：
  `pip install --force-reinstall --no-deps torch==2.9.0`
- Colab 以后升级 Python 不影响这个 venv；它固定使用 3.12。

### FlashAttention-2 装不上

- 确认装进了 `/content/verl_env`（cp312），不是 Colab 的 3.13。
- wheel 必须匹配 **torch 2.9.0 + CUDA 12.8**，预编译包链接见第 2 节；装完 `import flash_attn` 能打印 `2.8.3` 即成功。
- 装好后把第 1 节的 `USE_FLASH_ATTN=True` 并重跑第 1 节；否则训练一直走 SDPA。

### HuggingFace 401 / 403

- `Qwen/Qwen3-0.6B`、`Qwen/Qwen3-4B-Instruct-2507` 是公开模型，正常不会 401/403。
- 如果遇到下载失败，优先检查代理 / `HF_ENDPOINT`；国内网络可设置 `HF_ENDPOINT=https://hf-mirror.com`。
- 仍然失败时，可以手动把模型文件放到 `/content/models/Qwen3-0.6B` 和 `/content/models/Qwen3-4B-Instruct-2507`；`download_model()` 见到 `config.json` 会跳过下载。

### Qwen3 thinking 模式（重要）

- 教师 `Qwen3-4B-Instruct-2507` 是 **non-thinking** 模型，学生 rollout 也必须 `enable_thinking=False`，否则教师和学生在不同模式下打分，蒸馏目标会失真（2026-11-05 的 Qwen3-1.7B run 就是因此作废）。
- notebook 通过 `ENABLE_THINKING=False` 传给 `train_opd_sample.sh`，脚本会为 Qwen 模型加 `+data.apply_chat_template_kwargs.enable_thinking=False`。
- 评测单元也显式传 `--no-enable-thinking`，保证训练 / 评测一致。

### Ray / `/dev/shm` / `Bus error`

- 本 notebook 默认通过 `EXTRA_HYDRA_ARGS` 设置 `data.dataloader_num_workers=0`。
- 如果仍报 shared memory 错误：`Runtime → Restart session` 后重跑；不要在同一个 runtime 里反复启动 Ray。
- 可以用 `ray stop` 清理残留进程。

### 训练很慢 / 会话断开

- L4 上 40 步约 2–3 小时；Colab 免费版会因空闲断开，Pro 也可能在长时间无操作后断开。
- `SAVE_FREQ=20` 表示第 20 / 40 步各存一次；想更稳可以设成 10。
- 断开后 checkpoint 还在 `/content/outputs/...`，但 runtime 一旦被回收就没了；重要结果及时按第 8 节存 Drive。
- **新开一个 Colab 会话要重跑第 2 节的终端安装**（`/content/verl_env` 不跨会话保存），模型和输出同理。

### 为什么不用全词表 KL？

`opd.loss_type=reverse_kl` 需要为每个 response token 保留完整词表的 logits（V=151936），24GB 显存很容易 OOM。`sample_reverse_kl` 只对 rollout 采样到的 token 计算 KL（teacher 侧只缓存采样 token 的 log-prob），是本 notebook 的默认选择。

### 为什么教师/学生必须是同一个 tokenizer？

OPD 的 batch builder 只构造一份 `input_ids`，同时作为 `student_input_ids` 和 `teacher_input_ids`（见 `src/opd/batch_builder.py`）。词表不同会导致 teacher forward 的 token 语义完全错位。Qwen3 全系共用 151936 词表，所以 0.6B/1.7B/4B/8B 可以任意组合。

## 附：换成其他 Qwen3 模型

| 学生 | 教师 | 适用 |
|---|---|---|
| `Qwen/Qwen3-0.6B` | `Qwen/Qwen3-4B-Instruct-2507` | L4 24GB（默认，与本服务器一致） |
| `Qwen/Qwen3-0.6B` | `Qwen/Qwen3-8B-Instruct-2507` | A100 40GB（改 `TEACHER_ID`） |
| `Qwen/Qwen3-1.7B` | `Qwen/Qwen3-8B-Instruct-2507` | A100 80GB / H100 |

要求：

- **同一个 tokenizer**：Qwen3 全系共用 151936 词表，满足。
- 教师建议用 non-thinking 的 Instruct 版本（如 `Qwen3-*-Instruct-2507`），与本 notebook 的 boxed 指令格式一致。
- 教师只加载权重；prompt 始终由**学生 tokenizer** 渲染，所以教师不需要单独的 chat template。
- 教师 / 学生模式必须一致：教师是 non-thinking 时保持 `ENABLE_THINKING=False`（`train_opd_sample.sh` 会按模型名自动加 `enable_thinking` 参数，并在教师为 Instruct 且 thinking=True 时报错）。

---

## 附：本 notebook 与仓库脚本的对应关系

| 步骤 | 仓库脚本 |
|---|---|
| 数据转换 | `src/data/prepare_grpo_data.py` |
| 切 OPD 子集 | `src/data/split_opd_train.py` |
| 训练 | `scripts/opd/train_opd_sample.sh` → `python -m opd.main_opd` |
| 单 checkpoint 评测 | `scripts/eval/eval_ckpt_bench.py` |
| 曲线汇总 | `scripts/eval/collect_eval_curve.py`、`scripts/eval/analyze_eval_curve.py` |